# Step 3 — SQL Analytical Layer

SQL-based querying against the curated fact table and dimension tables built in Step 2.

**Scope:** Fleet-level KPI verification, delay and disruption analysis, and airline performance scoring. No data transformation or cleaning.

## 01 — Imports and DuckDB Setup

In [1]:
# -----------------------------------------
# Imports
# -----------------------------------------

import duckdb
import pandas as pd
from pathlib import Path


# -----------------------------------------
# Project Paths
# -----------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "Data"
PROCESSED_PATH = DATA_PATH / "Processed"
SQL_OUTPUT_PATH = PROCESSED_PATH / "sql_outputs"

SQL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("Processed data path:", PROCESSED_PATH)
print("SQL output path:", SQL_OUTPUT_PATH)

Processed data path: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\Data\Processed
SQL output path: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\Data\Processed\sql_outputs


In [4]:
FLIGHTS = PROCESSED_PATH / "fact_flights.parquet"

TABLE_PATHS = {
    "dim_airlines": PROCESSED_PATH / "dim_airlines.csv",
    "dim_airports": PROCESSED_PATH / "dim_airports.csv",
    "dim_cancel_codes": PROCESSED_PATH / "dim_cancellation_codes.csv",
    "dim_delay_driver": PROCESSED_PATH / "dim_delay_driver.csv",
}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

for df_name, path in TABLE_PATHS.items():
    if path.exists():
        globals()[df_name] = pd.read_csv(path)
        print(f"{df_name}: loaded {globals()[df_name].shape}")
    else:
        print(f"{df_name}: NOT FOUND")

if FLIGHTS.exists():
    flights = pd.read_parquet(FLIGHTS)
    print(f"flights: loaded {flights_df.shape}")
else:
    print("flights_df: NOT FOUND")

PROJECT_ROOT: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio
dim_airlines: loaded (14, 2)
dim_airports: loaded (322, 8)
dim_cancel_codes: loaded (4, 2)
dim_delay_driver: loaded (6, 3)
flights: loaded (5819079, 40)


## 02 — Schema Inspection

In [13]:
display(flights.info())

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 40 columns):
 #   Column                              Dtype   
---  ------                              -----   
 0   flight_date                         object  
 1   AIRLINE                             category
 2   FLIGHT_NUMBER                       Int32   
 3   ORIGIN_AIRPORT                      string  
 4   DESTINATION_AIRPORT                 string  
 5   SCHEDULED_DEPARTURE                 Int32   
 6   TAIL_NUMBER                         category
 7   scheduled_departure_hour            int8    
 8   time_band                           category
 9   day_of_week                         str     
 10  month                               str     
 11  CANCELLED                           Int8    
 12  CANCELLATION_REASON                 category
 13  DIVERTED                            Int8    
 14  DISTANCE                            Int16   
 15  SCHEDULED_TIME                      Int16  

None

## 03 — Fleet-Level KPI Queries

Verifies the KPI flags derived in Step 2 by computing each metric directly in SQL against the fact table.

### 03.1 On-Time Performance Rate (OTP15)

Counts eligible and on-time flights, then expresses on-time arrivals as a percentage of OTP15-eligible flights.

In [9]:
duckdb.sql("""
WITH otp15_counts AS (
    SELECT
        COUNT(*) FILTER (WHERE is_otp15_eligible = 1) AS otp15_eligible_flights,
        COUNT(*) FILTER (
            WHERE is_otp15_eligible = 1
              AND is_on_time_otp15 = 1
        ) AS on_time_otp15_flights
    FROM flights
)

SELECT
    otp15_eligible_flights,
    on_time_otp15_flights,
    ROUND(
        100.0 * on_time_otp15_flights
        / NULLIF(otp15_eligible_flights, 0),
        1
    ) AS otp15_rate_pct
FROM otp15_counts;
""").df()

,otp15_eligible_flights,on_time_otp15_flights,otp15_rate_pct
0,5714008,4690510,82.1


### 03.2 Average Arrival Delay

Computes the fleet-wide mean arrival delay in minutes across all flights.

In [26]:
duckdb.sql("""
SELECT
    ROUND(
        AVG(ARRIVAL_DELAY),
        1
    ) AS Average_Arrival_Delay_Minutes
FROM flights;
""").df()

,Average_Arrival_Delay_Minutes
0,4.4


### 03.3 Severe Delay Rate

Calculates the proportion of eligible flights with an arrival delay of 60 minutes or more.

In [14]:
duckdb.sql("""
WITH severe_delay_counts AS (
    SELECT
        COUNT(*) FILTER (WHERE is_otp15_eligible = 1) AS otp15_eligible_flights,
        COUNT(*) FILTER (
            WHERE is_severe_delay_sd60 = 1
              AND is_otp15_eligible = 1
        ) AS is_severe_delay_sd60_flights
    FROM flights
)

SELECT
    otp15_eligible_flights,
    is_severe_delay_sd60_flights,
    ROUND(
        100.0 * is_severe_delay_sd60_flights
        / NULLIF(otp15_eligible_flights, 0),
        1
    ) AS severe_delay_rate_pct
FROM severe_delay_counts;
""").df()

,otp15_eligible_flights,is_severe_delay_sd60_flights,severe_delay_rate_pct
0,5714008,325591,5.7


### 03.4 Cancellation Rate

Returns the total cancellation count and cancellation rate as a percentage of all scheduled flights.

In [17]:
duckdb.sql("""
WITH cancellation_counts AS (
    SELECT
        COUNT(*) AS total_flights,
        COUNT(*) FILTER (WHERE CANCELLED = 1) AS cancelled_flights
    FROM flights
)

SELECT
    total_flights,
    cancelled_flights,
    ROUND(
        100.0 * cancelled_flights / NULLIF(total_flights, 0),
        1
    ) AS cancellation_rate_pct
FROM cancellation_counts;
""").df()

,total_flights,cancelled_flights,cancellation_rate_pct
0,5819079,89884,1.5


### 03.5 Operational Disruption Rate

Measures the share of flights flagged as operationally disrupted — cancelled, diverted, or severely delayed (SD60).

In [20]:
duckdb.sql("""
WITH disruption_counts AS (
    SELECT
        COUNT(*) AS total_flights,
        COUNT(*) FILTER (WHERE operational_disruption_flag = 1) AS dirupted_flights
    FROM flights
)

SELECT
    total_flights,
    dirupted_flights,
    ROUND(
        100.0 * dirupted_flights / NULLIF(total_flights, 0),
        1
    ) AS disruption_rate_pct
FROM disruption_counts;
""").df()

,total_flights,dirupted_flights,disruption_rate_pct
0,5819079,430662,7.4


In [ ]:
duckdb.sql("""
WITH disruption_counts AS (
    SELECT
        COUNT(*) AS total_flights,
        COUNT(*) FILTER (WHERE operational_disruption_flag = 1) AS dirupted_flights
    FROM flights
)

SELECT
    total_flights,
    dirupted_flights,
    ROUND(
        100.0 * dirupted_flights / NULLIF(total_flights, 0),
        1
    ) AS disruption_rate_pct
FROM disruption_counts;
""").df()

## 04 — Airline Performance Scorecard

Ranks each carrier across all core KPIs — OTP15 rate, average arrival delay, severe delay rate, cancellation rate, and overall disruption rate — using a window-based performance rank.

In [19]:
duckdb.sql("""
WITH airline_performance AS (
    SELECT
        a.airline_name,

        COUNT(*) AS total_flights,

        COUNT(*) FILTER (
            WHERE f.CANCELLED = 0
        ) AS completed_flights,

        COUNT(*) FILTER (
            WHERE f.is_otp15_eligible = 1
        ) AS otp15_eligible_flights,

        COUNT(*) FILTER (
            WHERE f.is_otp15_eligible = 1
              AND f.is_on_time_otp15 = 1
        ) AS on_time_otp15_flights,

        AVG(f.ARRIVAL_DELAY) FILTER (
            WHERE f.CANCELLED = 0
        ) AS avg_arrival_delay_minutes,

        COUNT(*) FILTER (
            WHERE f.is_severe_delay_sd60 = 1
              AND f.CANCELLED = 0
        ) AS severe_delay_flights,

        COUNT(*) FILTER (
            WHERE f.CANCELLED = 1
        ) AS cancelled_flights,

        COUNT(*) FILTER (
            WHERE f.CANCELLED = 1
               OR f.is_severe_delay_sd60 = 1
        ) AS disrupted_flights

    FROM flights AS f
    LEFT JOIN dim_airlines AS a
        ON f.AIRLINE = a.airline_code

    GROUP BY
        a.airline_name
)

SELECT
    airline_name,
    total_flights,
    completed_flights,

    ROUND(
        100.0 * on_time_otp15_flights
        / NULLIF(otp15_eligible_flights, 0),
        2
    ) AS otp15_rate_pct,

    ROUND(avg_arrival_delay_minutes, 2) AS avg_arrival_delay_minutes,

    ROUND(
        100.0 * severe_delay_flights
        / NULLIF(completed_flights, 0),
        2
    ) AS severe_delay_rate_pct,

    ROUND(
        100.0 * cancelled_flights
        / NULLIF(total_flights, 0),
        2
    ) AS cancellation_rate_pct,

    ROUND(
        100.0 * disrupted_flights
        / NULLIF(total_flights, 0),
        2
    ) AS disruption_rate_pct,

    RANK() OVER (
        ORDER BY
            ROUND(
                100.0 * on_time_otp15_flights
                / NULLIF(otp15_eligible_flights, 0),
                2
            ) DESC,
            avg_arrival_delay_minutes ASC,
            ROUND(
                100.0 * cancellation_rate_pct,
                2
            ) ASC
    ) AS performance_rank

FROM airline_performance

ORDER BY
    performance_rank;
""").df()

,airline_name,total_flights,completed_flights,otp15_rate_pct,avg_arrival_delay_minutes,severe_delay_rate_pct,cancellation_rate_pct,disruption_rate_pct,performance_rank
0,Hawaiian Airlines Inc.,76272,76101,89.47,2.02,1.46,0.22,1.68,1
1,Alaska Airlines Inc.,172521,171852,87.66,-0.98,2.94,0.39,3.32,2
2,Delta Air Lines Inc.,875881,872057,87.00,0.19,4.04,0.44,4.46,3
3,American Airlines Inc.,725984,715065,82.43,3.45,5.44,1.50,6.86,4
4,US Airways Inc.,198715,194648,82.02,3.71,4.65,2.05,6.60,5
5,Skywest Airlines Inc.,588353,578393,82.00,5.85,6.04,1.69,7.63,6
6,Southwest Airlines Co.,1261855,1245812,81.70,4.37,5.08,1.27,6.28,7
7,Virgin America,61903,61369,81.47,4.74,5.79,0.86,6.60,8
8,Atlantic Southeast Airlines,571977,556746,81.01,6.59,6.61,2.66,9.10,9
9,United Air Lines Inc.,515723,509150,80.05,5.43,7.21,1.27,8.39,10


## Step 3 Summary

Step 3 verifies the KPI flags from Step 2 and delivers a ranked airline performance scorecard using DuckDB SQL.

**SQL outputs directory:** `data/processed/sql_outputs/`

---

> **Work in progress** — additional queries (route-level analysis, delay driver breakdown, monthly trends) will be added here.